# 02 — Scratch CNN Baseline and Bayesian Hyperparameter Selection

**Question:** Which scratch-CNN architecture and training settings maximize **validation Macro F1**, independently within each regime?

Each regime starts with a **fresh randomly initialized model**, its own training and validation partitions, and its own Bayesian search. The test sets are not used for search or early stopping. The best checkpoint for each run is selected using the **first epoch** reaching that run's highest validation Macro F1; the best trial is selected using the same explicitly named metric. Only then is the chosen model evaluated on its held-out test images.

The cells below are executable research code, **not downloaded results**. For routine authentication-free experimentation, Optuna TPE provides Bayesian sequential search over the same recorded parameter ranges. For an actual *new* W&B Bayesian sweep, set `MONREADER_TUNER=wandb_bayes` before running Notebook 01 and authenticate privately; this uses W&B's `method="bayes"`. These are different search optimizers and need not propose identical trial sequences.

In [ ]:
from pathlib import Path
import sys, json, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path.cwd().resolve()
sys.path.insert(0,str(ROOT/'generated'))
import monreader_methods as mr
settings=mr.load_settings(ROOT)
device=mr.setup(settings['seed'],settings['smoke'])
print('Device:',device,' | Tuner:',settings['tuner_backend'])
print('MAX_EPOCHS:',settings['max_epochs'],' | Patience:',settings['patience'])
print('Qualifying trial target per regime:',settings['qualifying_trials'])

## Architecture, loss and optimization

The scratch network applies configurable blocks of convolution, batch normalization, ReLU and pooling, followed by adaptive global pooling, dropout and a **one-logit** binary classification head. Binary cross-entropy with logits avoids a separate sigmoid in the training loss. Training uses threshold 0.50 for validation decisions, while prediction probabilities are sigmoid-transformed. The notebook caps expanded convolution channels at 256 for practical memory use; this cap is documented rather than concealed.

In the full research profile, each trial can run at most **50 epochs**, with early stopping patience **4** for new trials. Hardware failures are recorded rather than silently treated as successes. The smoke profile reduces epochs, depth and number of trials solely to test the same execution pathway on synthetic files.

**Bayesian search (research profile):** optimizer `{Adam, AdamW, SGD, RMSprop}`; learning rate `1e-5..1e-2` log; batch size `8..256`; zero/positive weight decay with positive value `1e-6..1e-2` log; dropout `0..0.5`; depth `1..7`; and starting filters `8..64`. The goal is **30 qualifying nonzero finite validation Macro F1 trials per regime**, excluding failed attempts. No result is entered in advance.

## Optional audit of one fixed baseline

The configuration below is a deliberately defined **baseline**, not a tuned winner. It can be used to obtain a controlled comparison against the selected model, without peeking at any test result. Model parameter counts are computed directly from the instantiated architecture.

In [ ]:
baseline_config=dict(depth=3,starting_filters=16,dropout=.20,optimizer='adam',learning_rate=1e-3,
                     batch_size=16,weight_decay=0.)
baseline=mr.ScratchCNN(baseline_config['depth'],baseline_config['starting_filters'],baseline_config['dropout'])
print('Baseline trainable parameters:',sum(p.numel() for p in baseline.parameters() if p.requires_grad))
print('Baseline configuration:',baseline_config)
del baseline

## Four independent Bayesian searches

This is the actual experiment. It generates new trial configurations and epoch histories, and reports both **best validation Macro F1** and **final held-out Macro F1**. The latter must never be used to choose a trial or checkpoint. Candidate tables, selected checkpoints and per-frame validation/test predictions are saved locally for later notebooks. In W&B mode, brand-new sweeps are created under the researcher's account; in default Optuna mode, no network credentials are required.

In [ ]:
all_scratch=[]
for regime in mr.REGIMES:
    frame=pd.read_csv(Path(settings['out'])/'splits'/f'{regime}_split.csv')
    if settings['tuner_backend']=='optuna':
        result=mr.tune_one_regime(frame,regime,settings,device)
    elif settings['tuner_backend']=='wandb_bayes':
        result=mr.tune_one_regime_wandb(frame,regime,settings,device)
    else:raise ValueError('Unsupported tuner: '+str(settings['tuner_backend']))
    all_scratch.append(result)
metrics=pd.DataFrame(all_scratch)
metrics.to_csv(Path(settings['out'])/'scratch_heldout_metrics.csv',index=False)
display(metrics[['regime','val_validation_macro_f1','test_macro_f1','test_accuracy']])

## Plot the evidence rather than paste old sweep results

The following plot reads the trials and training histories just generated by the current run. It highlights the best selected epoch and displays the observed validation objective. In a new experiment the results may differ from another researcher's runs: this is expected and scientifically acceptable.

In [ ]:
fig,axes=plt.subplots(2,4,figsize=(15,6))
for j,regime in enumerate(mr.REGIMES):
    folder=Path(settings['out'])/'scratch'/regime
    tr=pd.read_csv(folder/'bayesian_trials.csv')
    tr=tr[tr.get('status',pd.Series('finished',index=tr.index)).eq('finished')]
    axes[0,j].scatter(tr['trial'],tr['validation_macro_f1'],s=15)
    axes[0,j].set(title=f'{regime}: Bayesian trials',xlabel='Trial',ylabel='Validation Macro F1')
    hist=pd.read_csv(folder/'selected_training_history.csv')
    axes[1,j].plot(hist['epoch'],hist['train_loss'],label='Train loss')
    axes[1,j].plot(hist['epoch'],hist['validation_loss'],label='Validation loss')
    axes[1,j].set(title=f'{regime}: selected training',xlabel='Epoch',ylabel='BCE loss')
axes[1,0].legend()
fig.tight_layout();out=Path(settings['out'])/'figures'/'bayesian_tuning_and_training.png';fig.savefig(out,dpi=150);plt.show()
print('Saved:',out)

## Interpretation and handoff

Model comparison must not confuse (a) per-epoch **validation Macro F1** used during Bayesian selection and (b) **held-out Macro F1** used as the final result. Each of the four selected scratch-CNN models is saved under its own regime directory. Notebook 03 will independently fine-tune ResNet18 models using the corresponding splits; it will not continue training an O model on S, T or ST.